# Unmapped: run your data through Adaption

This notebook checks your dataset, sends it to Adaption's **Adaptive Data**, and saves the result with a data card you can publish.

**How to use it:** click a grey code cell, then press **Shift + Enter** to run it. Go from top to bottom. Each cell says what it does first.

**You need** your Adaption API key. Your $800 in credits sits on the Adaption account that uses the same email you registered with on Build54.

Part of the [Unmapped starter kit](https://github.com/build54Africa/unmapped-starter-kit).

**1. Install.** This adds the Adaption library and the tools that read tables. It takes about a minute the first time.

In [ ]:
import os, subprocess, sys

if os.getenv("UNMAPPED_NOTEBOOK_TEST") != "1":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "adaption==0.17.1", "pandas", "openpyxl"])
print("Ready.")

**2. Your API key.** In Colab, click the key icon on the left, add a secret called ADAPTION_API_KEY, paste your key and switch on notebook access. This cell reads it without showing it. Never share your key, or a screenshot of it.

In [ ]:
import getpass

try:
    from google.colab import userdata
    API_KEY = userdata.get("ADAPTION_API_KEY")
except Exception:
    API_KEY = os.getenv("ADAPTION_API_KEY") or getpass.getpass("Adaption API key (hidden): ")
if not API_KEY:
    raise ValueError("No API key found. Add ADAPTION_API_KEY to Colab secrets (the key icon on the left).")
print("Key loaded. It stays hidden.")

**3. Choose your data.** Upload your own file, paste a link to one, or start from the sample for your impact area. Change the settings in this cell, then run it.

In [ ]:
DATA_SOURCE = "upload"   # "upload" opens a file picker, "url" uses DATA_URL, "sample" uses SAMPLE_AREA
DATA_URL = ""            # a link to a CSV, Excel or JSON file
SAMPLE_AREA = "health"   # health, food, climate, learning or money
COUNTRIES = ["Nigeria", "Kenya", "Rwanda"]  # tables that list every country keep only these

SAMPLES = {
    "health": {
        "name": "Rwanda Healthsites",
        "url": "https://data.humdata.org/dataset/67ca8390-678a-4eb0-8c06-a12d86df1033/resource/752482cc-e7a0-4898-8653-113d0373fba6/download/rwanda.csv",
        "options": {}
    },
    "food": {
        "name": "Kenya - Food Prices",
        "url": "https://data.humdata.org/dataset/e0d3fba6-f9a2-45d7-b949-140c455197ff/resource/517ee1bf-2437-4f8c-aa1b-cb9925b9d437/download/wfp_food_prices_ken.csv",
        "options": {}
    },
    "climate": {
        "name": "NASA POWER daily weather near Nairobi",
        "url": "https://power.larc.nasa.gov/api/temporal/daily/point?parameters=T2M_MAX,PRECTOTCORR&community=AG&longitude=36.817&latitude=-1.286&start=20000101&end=20251231&format=CSV",
        "options": {
            "skiprows": 10
        }
    },
    "learning": {
        "name": "Primary school completion rate",
        "url": "https://ourworldindata.org/grapher/school-completion.csv?v=1&csvType=full&useColumnShortNames=false",
        "options": {}
    },
    "money": {
        "name": "Mobile money account usage",
        "url": "https://ourworldindata.org/grapher/mobile-money-account-usage.csv?v=1&csvType=full&useColumnShortNames=false",
        "options": {}
    }
}

**4. The reader.** One function that opens CSV, Excel and JSON files the same way.

In [ ]:
from pathlib import Path
from urllib.parse import urlparse
import pandas as pd


def read_table(source, **options):
    if str(source).startswith(("http://", "https://")):
        # Some data sites refuse downloads that do not say who is asking.
        options.setdefault("storage_options", {"User-Agent": "Mozilla/5.0 (Build54 Unmapped starter notebook)"})
    suffix = Path(urlparse(str(source)).path).suffix.lower()
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(source, **options)
    if suffix in {".json", ".jsonl"}:
        try:
            return pd.read_json(source, **options)
        except ValueError:
            return pd.read_json(source, lines=True, **options)
    return pd.read_csv(source, **options)  # CSV, and links that do not end in a file name

**5. Load it.** This opens your data as a table called df and shows the first rows.

In [ ]:
options = {}
test_path = os.getenv("UNMAPPED_TEST_DATASET")
if test_path:
    dataset_source = test_path
elif DATA_SOURCE == "url":
    dataset_source = DATA_URL
elif DATA_SOURCE == "sample":
    chosen = SAMPLES[SAMPLE_AREA]
    dataset_source, options = chosen["url"], chosen["options"]
    print("Sample:", chosen["name"])
else:
    try:
        from google.colab import files
        dataset_source = next(iter(files.upload()))
    except Exception as error:
        raise RuntimeError("Upload works in Colab. Elsewhere, use DATA_SOURCE = 'url' or 'sample'.") from error
if not dataset_source:
    raise ValueError("Add a link in DATA_URL, or choose 'upload' or 'sample'.")
df = read_table(dataset_source, **options)
for country_column in ("Entity", "Country", "country", "country_name"):
    if country_column in df.columns and df[country_column].isin(COUNTRIES).any():
        df = df[df[country_column].isin(COUNTRIES)].reset_index(drop=True)
        print(f"Kept rows for {', '.join(COUNTRIES)} only.")
        break
if df.empty or len(df.columns) == 0:
    raise ValueError("The table is empty. Check the file or the link.")
print(f"Loaded {len(df):,} rows and {len(df.columns):,} columns.")
display(df.head())

**6. Look at what you have.** How many rows and columns, what type each column is, and how many blanks each one has.

In [ ]:
duplicate_rows = int(df.astype(str).duplicated().sum())
summary = pd.DataFrame({"": ["rows", "columns", "duplicate rows"], "count": [len(df), len(df.columns), duplicate_rows]})
columns = pd.DataFrame({"column": df.columns, "type": df.dtypes.astype(str).values, "blank values": df.isna().sum().values})
display(summary)
display(columns)

**7. What is missing.** The columns with blanks, and what share of rows they miss. These gaps are often where your project starts.

In [ ]:
missing = columns.loc[columns["blank values"] > 0, ["column", "blank values"]].copy()
missing["share missing (%)"] = (missing["blank values"] / max(len(df), 1) * 100).round(1)
if missing.empty:
    print("No blank values. Look for what the table does not cover at all: places, groups, dates.")
else:
    print("These columns have gaps:")
    display(missing.sort_values("share missing (%)", ascending=False))

**8. Check for personal data.** Your table goes to Adaption next. Remove names of private people, phone numbers, emails, exact home locations and ID numbers first. This cell flags columns that might hold them. Check the list, fix your data if needed, then change the setting to True and run the cell again.

In [ ]:
I_CHECKED_FOR_PERSONAL_DATA = False  # change to True once your data has no personal details

import re

NAME_HINTS = ("name", "phone", "mobile", "email", "address", "national_id", "id_number", "birth")
EMAIL = re.compile(r"[^@\s]+@[^@\s]+\.[a-z]{2,}", re.IGNORECASE)
PHONE = re.compile(r"\+?\d[\d\s-]{8,}\d")
flags = []
for column in df.columns:
    reasons = []
    if any(hint in str(column).lower() for hint in NAME_HINTS):
        reasons.append("its name")
    if not pd.api.types.is_numeric_dtype(df[column]):
        values = df[column].dropna().astype(str).head(500)
        if values.str.contains(EMAIL).any():
            reasons.append("email addresses")
        if values.str.contains(PHONE).any():
            reasons.append("possible phone numbers")
    if reasons:
        flags.append({"column": column, "check it because of": ", ".join(reasons)})
if flags:
    print("Check these columns. Names of places or organisations are fine; details about private people are not.")
    display(pd.DataFrame(flags))
else:
    print("Nothing obvious found. Still look through your data yourself.")
print("Checked:", I_CHECKED_FOR_PERSONAL_DATA)

**9. Prepare the upload.** Adaption takes CSV files, so this saves a CSV copy of your table.

In [ ]:
UPLOAD_PATH = "unmapped_input.csv"
df.to_csv(UPLOAD_PATH, index=False)
print(f"Saved {UPLOAD_PATH} with {len(df):,} rows.")

**10. Tell Adaption what to make.** Adaptive Data turns each row into an example that teaches AI about your area: an instruction, the row's details, and an answer it writes from them. Leave PROMPT_COLUMN empty to use the instruction below on every row, or set it if your table already has a question column. Keep MAX_ROWS small for your first run.

In [ ]:
UNIVERSAL_PROMPT = (
    "Using only the details in this row, write two or three plain sentences that someone living there "
    "could act on: what it is, where it is, and the numbers that matter."
)
PROMPT_COLUMN = ""        # a column with a question or instruction on each row, if you have one
COMPLETION_COLUMN = ""    # a column with the answer on each row, if you have one
CONTEXT_COLUMNS = []      # the columns that describe each row; empty means all of them
MAX_ROWS = min(len(df), 100)

for name in [PROMPT_COLUMN, COMPLETION_COLUMN, *CONTEXT_COLUMNS]:
    if name and name not in df.columns:
        raise ValueError(f"There is no column called {name!r}. Check the spelling against the table above.")
context = CONTEXT_COLUMNS or [str(column) for column in df.columns if str(column) not in (PROMPT_COLUMN, COMPLETION_COLUMN)]
if PROMPT_COLUMN:
    column_mapping = {"prompt": PROMPT_COLUMN, "context": context}
else:
    column_mapping = {"universal_prompt": UNIVERSAL_PROMPT, "context": context}
if COMPLETION_COLUMN:
    column_mapping["completion"] = COMPLETION_COLUMN
print(f"Adaption will work on {MAX_ROWS:,} rows using {len(context)} columns as each row's details.")

**11. Send it to Adaption.** This uploads the CSV and waits until Adaption has counted the rows.

In [ ]:
import time
from adaption import Adaption

if not I_CHECKED_FOR_PERSONAL_DATA:
    raise RuntimeError("Check your data for personal details first, then set I_CHECKED_FOR_PERSONAL_DATA = True in step 8.")
client = Adaption(api_key=API_KEY)
dataset_id = client.datasets.upload_file(UPLOAD_PATH).dataset_id
while True:
    status = client.datasets.get_status(dataset_id)
    if status.status == "failed":
        raise RuntimeError(f"Adaption could not read the file: {status.error_data}")
    if status.row_count is not None:
        break
    time.sleep(5)
print(f"Adaption has your data: {status.row_count:,} rows. Dataset id: {dataset_id}")

**12. See the price first.** This asks Adaption how many credits the run would use. Nothing is spent yet.

In [ ]:
estimate = client.datasets.adapt(
    dataset_id,
    column_mapping=column_mapping,
    job_specification={"max_rows": MAX_ROWS},
    estimate=True,
)
print(f"Estimated credits: {estimate.estimated_credits_consumed}")
print(f"Estimated time: about {estimate.estimated_minutes} minutes")
print("Nothing has been spent yet.")

**13. The spending switch.** Happy with the estimate? Change False to True and run this cell, then run the next one.

In [ ]:
RUN_FOR_REAL = False
print("Ready to spend credits:", RUN_FOR_REAL)

**14. Run it.** Adaption adapts your rows. This cell waits until it finishes, then saves the result as adapted.csv.

In [ ]:
adapted_df = None
run_status = "not run yet"
if RUN_FOR_REAL:
    client.datasets.adapt(dataset_id, column_mapping=column_mapping, job_specification={"max_rows": MAX_ROWS})
    final = client.datasets.wait_for_completion(dataset_id, timeout=3600)
    run_status = final.status
    if run_status != "succeeded":
        raise RuntimeError(f"The run ended as {run_status}: {final.error_data}")
    client.datasets.download(dataset_id, file_format="csv").write_to_file("adapted.csv")
    adapted_df = pd.read_csv("adapted.csv")
    print(f"Saved adapted.csv with {len(adapted_df):,} rows.")
    display(adapted_df.head())
else:
    print("Not run yet: nothing was spent. Turn on RUN_FOR_REAL in step 13 when you are ready.")

**15. Before and after.** Rows, columns and blank values in your table and in Adaption's result.

In [ ]:
after = adapted_df if adapted_df is not None else None
comparison = pd.DataFrame({
    "": ["rows", "columns", "blank values"],
    "before": [len(df), len(df.columns), int(df.isna().sum().sum())],
    "after": [len(after), len(after.columns), int(after.isna().sum().sum())] if after is not None else [None] * 3,
})
display(comparison)

**16. Your data card.** This writes DATA_CARD.md with what the notebook knows. Open it and replace every TODO before you publish.

In [ ]:
dataset_name = Path(urlparse(str(dataset_source)).path).stem or "my dataset"
gaps = "none" if missing.empty else "; ".join(
    f"{row['column']} ({row['share missing (%)']}% blank)" for _, row in missing.iterrows()
)
rows_after = len(adapted_df) if adapted_df is not None else "not run yet"
card = f"""# Data card: {dataset_name}

## In one line
TODO: what this dataset is and who it helps, in one sentence.

## Sources and permissions
- Source: {dataset_source}
- Licence: TODO: copy it from the source page.
- Permission: TODO: confirm the licence lets you share a changed copy.

## What I changed
- Checked the data: {len(df):,} rows, {len(df.columns)} columns, {duplicate_rows} duplicate rows. Blank values: {gaps}.
- Ran Adaption Adaptive Data on up to {MAX_ROWS:,} rows with this setup: {column_mapping}. Result: {run_status}.
- TODO: any cleaning, joining or new columns you added yourself.

## What it covers
- Places: TODO
- Time period: TODO
- Rows after Adaptive Data: {rows_after}

## What is still missing or biased
- TODO: places, groups, dates or languages the data misses.
- TODO: anything that could lead someone to the wrong conclusion.

## Personal data check
- [ ] No names of private people
- [ ] No phone numbers or email addresses
- [ ] No exact home locations
- [ ] No national, school, medical or other ID numbers

## How to cite
TODO: your team, the dataset name, its link and the year.
"""
Path("DATA_CARD.md").write_text(card, encoding="utf-8")
print("Saved DATA_CARD.md. Replace every TODO before you publish it.")

**17. Save your files.** In Colab this downloads adapted.csv and DATA_CARD.md to your computer.

In [ ]:
saved = [name for name in ("adapted.csv", "DATA_CARD.md") if Path(name).exists()]
try:
    from google.colab import files
    for name in saved:
        files.download(name)
except Exception:
    print("Your files are next to this notebook:", ", ".join(saved))

## Next steps

1. Finish the TODOs in DATA_CARD.md.
2. Publish adapted.csv with the data card on Kaggle or Hugging Face, under a licence the source allows.
3. Open your data in Adaption Visualize and find one insight worth acting on.
4. Back on Build54, paste the link to your copy of this notebook (in Colab: Share, then "Anyone with the link"), your dataset link and your insight into the tasks.

**If something goes wrong**
- *Key not working:* create a new key in Adaption under Settings, then API keys, and update the Colab secret.
- *No credits:* credits go to the Adaption account that uses your Build54 registration email. Sign in to Adaption with that email.
- *Big file:* lower MAX_ROWS, or keep only the columns you need with CONTEXT_COLUMNS.
- *Stuck:* contact the Build54 team at [build54.com/contact](https://build54.com/contact) with the error message. Never send your key.

[Back to the starter kit](https://github.com/build54Africa/unmapped-starter-kit)